# Exploratory Data Analysis - NASA Battery Dataset

**Project:** Battery SOC Estimation using Machine Learning  
**Affiliation:** IISER Bhopal Internship - Li-ion Battery SOC/SOH Estimation  
**Dataset:** NASA Prognostics Center of Excellence (PCoE) Battery Dataset  

---

This notebook performs a comprehensive exploratory data analysis (EDA) of Li-ion battery cycling data. We examine voltage, current, and temperature profiles across charge/discharge cycles, analyze capacity degradation trends, and identify correlations between measured features and the State of Charge (SOC).

## 1. Background: Li-ion Batteries, SOC, and SOH

### Lithium-Ion Batteries
Lithium-ion batteries are the dominant energy storage technology in electric vehicles, portable electronics, and grid-scale storage. Understanding their electrochemical behavior through data-driven methods is essential for safe and efficient operation.

### State of Charge (SOC)
SOC represents the remaining charge in the battery as a fraction of its current capacity:

$$\text{SOC}(t) = \text{SOC}(t_0) - \frac{1}{Q_{nom}} \int_{t_0}^{t} I(\tau) \, d\tau$$

where $Q_{nom}$ is the nominal capacity and $I(t)$ is the current (positive for discharge). Accurate SOC estimation is critical for battery management systems (BMS) to prevent overcharge/overdischarge.

### State of Health (SOH)
SOH quantifies the overall degradation of a battery compared to its initial condition:

$$\text{SOH} = \frac{C_{current}}{C_{nominal}} \times 100\%$$

A battery is typically considered to have reached end-of-life (EOL) when SOH drops below 80%.

### NASA Battery Dataset
The NASA PCoE dataset contains charge, discharge, and impedance measurements for multiple 18650 Li-ion cells (B0005, B0006, B0007, B0018) cycled at room temperature. Discharge was carried out at a 2A constant current until voltage dropped to specified cutoff levels. The cells exhibited capacity fade over hundreds of cycles, providing ground truth for SOH and degradation modeling.

## 2. Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path so we can import from src/
sys.path.insert(0, '..')

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('deep')

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 3. Load Data

We attempt to load the NASA battery dataset from the `data/` directory. If the real dataset is not available, we generate realistic synthetic battery cycling data using the project's `data_loader` module. The synthetic data simulates constant-current discharge with capacity degradation, voltage profiles based on an OCV-SOC model, and temperature rise during discharge.

In [ ]:
from src.data_loader import (
    generate_synthetic_battery_data,
    load_csv_battery_data,
    clean_battery_data,
    compute_soc_coulomb_counting,
    extract_cycle_capacities,
)

# Attempt to load real data; fall back to synthetic
data_dir = os.path.join('..', 'data')
processed_path = os.path.join(data_dir, 'processed', 'battery_processed.csv')

if os.path.exists(processed_path):
    print(f"Loading processed data from {processed_path}")
    df = pd.read_csv(processed_path)
else:
    try:
        df = load_csv_battery_data(data_dir)
        if len(df) == 0:
            raise FileNotFoundError("No CSV files found")
        df = clean_battery_data(df)
        df = compute_soc_coulomb_counting(df)
        print(f"Loaded real battery data: {len(df)} records")
    except (FileNotFoundError, Exception) as e:
        print(f"Real data not available ({e}). Generating synthetic battery data...")
        df = generate_synthetic_battery_data(
            n_cycles=200,
            points_per_cycle=500,
            nominal_capacity=2.0,
            degradation_rate=0.001,
            seed=42,
        )
        print(f"Generated synthetic data: {len(df)} records, {df['cycle'].nunique()} cycles")

# Standardize column names for the rest of the notebook
col_map = {
    'cycle': 'cycle_number',
    'time': 'time_s',
    'voltage': 'voltage_V',
    'current': 'current_A',
    'temperature': 'temperature_C',
}
df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

# Compute derived columns if not present
if 'SOC_pct' not in df.columns and 'soc' in df.columns:
    df['SOC_pct'] = df['soc'] * 100.0

# Compute capacity per cycle
if 'capacity_Ah' not in df.columns:
    df['capacity_Ah'] = np.nan
    for cyc in df['cycle_number'].unique():
        mask = df['cycle_number'] == cyc
        cycle_data = df.loc[mask]
        if len(cycle_data) >= 2:
            dt = cycle_data['time_s'].diff().fillna(0).values
            current = np.abs(cycle_data['current_A'].values)
            cum_ah = np.cumsum(current * dt) / 3600.0
            df.loc[mask, 'capacity_Ah'] = cum_ah

print(f"\nDataset ready: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Cycles: {df['cycle_number'].nunique()} (from {df['cycle_number'].min()} to {df['cycle_number'].max()})")

## 4. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Descriptive statistics for numeric columns
numeric_cols = ['voltage_V', 'current_A', 'temperature_C', 'capacity_Ah', 'SOC_pct']
available_cols = [c for c in numeric_cols if c in df.columns]
df[available_cols].describe().round(4)

In [ ]:
# First few rows
df.head(10)

## 5. Single Cycle Visualization

We examine the voltage, current, and temperature profiles for a single discharge cycle to understand the typical measurement patterns.

In [ ]:
# Select a representative early cycle
sample_cycle = df['cycle_number'].unique()[5]  # 6th cycle
cycle_data = df[df['cycle_number'] == sample_cycle].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Voltage
axes[0].plot(cycle_data['time_s'], cycle_data['voltage_V'], color='#2196F3', linewidth=1.5)
axes[0].set_ylabel('Voltage (V)', fontsize=12)
axes[0].set_title(f'Single Discharge Cycle Profile (Cycle {sample_cycle})', fontsize=14, fontweight='bold')
axes[0].axhline(y=2.7, color='red', linestyle='--', alpha=0.5, label='Cutoff (2.7V)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Current
axes[1].plot(cycle_data['time_s'], cycle_data['current_A'], color='#FF9800', linewidth=1.5)
axes[1].set_ylabel('Current (A)', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Temperature
axes[2].plot(cycle_data['time_s'], cycle_data['temperature_C'], color='#F44336', linewidth=1.5)
axes[2].set_ylabel('Temperature (\u00b0C)', fontsize=12)
axes[2].set_xlabel('Time (s)', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Cycle {sample_cycle}: {len(cycle_data)} data points")
print(f"  Voltage range: {cycle_data['voltage_V'].min():.3f} - {cycle_data['voltage_V'].max():.3f} V")
print(f"  Current range: {cycle_data['current_A'].min():.3f} - {cycle_data['current_A'].max():.3f} A")
print(f"  Temperature range: {cycle_data['temperature_C'].min():.2f} - {cycle_data['temperature_C'].max():.2f} \u00b0C")

## 6. Charge vs Discharge Analysis

We overlay voltage curves from multiple cycles to visualize how the discharge profile evolves with aging. Early cycles should show higher capacity (longer discharge time) compared to later cycles.

In [ ]:
# Select cycles at different stages of life
all_cycles = sorted(df['cycle_number'].unique())
n_total = len(all_cycles)
selected_indices = [0, n_total // 5, 2 * n_total // 5, 3 * n_total // 5, 4 * n_total // 5, n_total - 1]
selected_cycles = [all_cycles[i] for i in selected_indices]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Voltage vs. Time
cmap = plt.cm.viridis(np.linspace(0, 1, len(selected_cycles)))
for idx, cyc in enumerate(selected_cycles):
    cyc_data = df[df['cycle_number'] == cyc]
    # Normalize time to start from 0
    t_norm = cyc_data['time_s'].values - cyc_data['time_s'].values[0]
    axes[0].plot(t_norm, cyc_data['voltage_V'].values, color=cmap[idx],
                 linewidth=1.5, label=f'Cycle {cyc}')

axes[0].set_xlabel('Time (s)', fontsize=12)
axes[0].set_ylabel('Voltage (V)', fontsize=12)
axes[0].set_title('Discharge Voltage Curves at Different Cycles', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Right: Voltage vs. SOC
if 'SOC_pct' in df.columns:
    for idx, cyc in enumerate(selected_cycles):
        cyc_data = df[df['cycle_number'] == cyc]
        if cyc_data['SOC_pct'].notna().sum() > 0:
            axes[1].plot(cyc_data['SOC_pct'].values, cyc_data['voltage_V'].values,
                         color=cmap[idx], linewidth=1.5, label=f'Cycle {cyc}')

    axes[1].set_xlabel('SOC (%)', fontsize=12)
    axes[1].set_ylabel('Voltage (V)', fontsize=12)
    axes[1].set_title('Voltage vs SOC at Different Cycles', fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].invert_xaxis()  # SOC decreases during discharge
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Capacity Fade Over Cycles

Capacity fade is the primary indicator of battery degradation (SOH). We compute the discharge capacity for each cycle and plot the trend over the battery's lifetime.

In [ ]:
# Compute max discharge capacity per cycle
cycle_caps = df.groupby('cycle_number').agg(
    max_capacity_Ah=('capacity_Ah', 'max'),
    mean_temp=('temperature_C', 'mean'),
    min_voltage=('voltage_V', 'min'),
    max_voltage=('voltage_V', 'max'),
    n_points=('voltage_V', 'count'),
).reset_index()

# SOH computation
nominal_cap = cycle_caps['max_capacity_Ah'].max()
cycle_caps['SOH_pct'] = (cycle_caps['max_capacity_Ah'] / nominal_cap) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Capacity fade
axes[0].plot(cycle_caps['cycle_number'], cycle_caps['max_capacity_Ah'],
             'o-', color='#1976D2', markersize=3, linewidth=1.2, alpha=0.8)
axes[0].set_xlabel('Cycle Number', fontsize=12)
axes[0].set_ylabel('Discharge Capacity (Ah)', fontsize=12)
axes[0].set_title('Capacity Fade Over Cycles', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# SOH trend
axes[1].plot(cycle_caps['cycle_number'], cycle_caps['SOH_pct'],
             'o-', color='#388E3C', markersize=3, linewidth=1.2, alpha=0.8)
axes[1].axhline(y=80, color='red', linestyle='--', linewidth=1.5, label='EOL Threshold (80%)')
axes[1].set_xlabel('Cycle Number', fontsize=12)
axes[1].set_ylabel('State of Health (%)', fontsize=12)
axes[1].set_title('SOH Degradation Trend', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Nominal capacity: {nominal_cap:.4f} Ah")
print(f"Final capacity: {cycle_caps['max_capacity_Ah'].iloc[-1]:.4f} Ah")
print(f"Capacity retention: {cycle_caps['SOH_pct'].iloc[-1]:.1f}%")
print(f"Capacity loss per cycle: {(nominal_cap - cycle_caps['max_capacity_Ah'].iloc[-1]) / len(cycle_caps) * 1000:.3f} mAh/cycle")

## 8. Temperature Effects

Temperature significantly affects both battery performance and degradation. We examine how temperature correlates with SOC estimation accuracy and capacity fade.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature distribution across all cycles
axes[0, 0].hist(df['temperature_C'].dropna(), bins=50, color='#F44336', alpha=0.7, edgecolor='white')
axes[0, 0].set_xlabel('Temperature (\u00b0C)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Temperature Distribution', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Temperature vs SOC
if 'SOC_pct' in df.columns:
    sample = df.dropna(subset=['SOC_pct', 'temperature_C']).sample(
        n=min(5000, len(df)), random_state=42
    )
    scatter = axes[0, 1].scatter(
        sample['SOC_pct'], sample['temperature_C'],
        c=sample['cycle_number'], cmap='viridis', alpha=0.3, s=5
    )
    plt.colorbar(scatter, ax=axes[0, 1], label='Cycle Number')
    axes[0, 1].set_xlabel('SOC (%)', fontsize=11)
    axes[0, 1].set_ylabel('Temperature (\u00b0C)', fontsize=11)
    axes[0, 1].set_title('Temperature vs SOC (colored by cycle)', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)

# Average temperature per cycle
axes[1, 0].plot(cycle_caps['cycle_number'], cycle_caps['mean_temp'],
                'o-', color='#FF5722', markersize=3, linewidth=1.0, alpha=0.8)
axes[1, 0].set_xlabel('Cycle Number', fontsize=11)
axes[1, 0].set_ylabel('Mean Temperature (\u00b0C)', fontsize=11)
axes[1, 0].set_title('Average Temperature per Cycle', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Temperature vs Capacity
axes[1, 1].scatter(cycle_caps['mean_temp'], cycle_caps['max_capacity_Ah'],
                   c=cycle_caps['cycle_number'], cmap='coolwarm', alpha=0.7, s=20)
axes[1, 1].set_xlabel('Mean Temperature (\u00b0C)', fontsize=11)
axes[1, 1].set_ylabel('Discharge Capacity (Ah)', fontsize=11)
axes[1, 1].set_title('Capacity vs Temperature', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Feature Distributions

We examine the distributions of the primary measurement features to check for skewness, outliers, and the need for normalization before ML modeling.

In [ ]:
plot_cols = ['voltage_V', 'current_A', 'temperature_C']
available_plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(1, len(available_plot_cols), figsize=(5 * len(available_plot_cols), 5))
if len(available_plot_cols) == 1:
    axes = [axes]

colors = ['#2196F3', '#FF9800', '#F44336']
labels = ['Voltage (V)', 'Current (A)', 'Temperature (\u00b0C)']

for i, col in enumerate(available_plot_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=60, color=colors[i], alpha=0.7, edgecolor='white', density=True)
    
    # Overlay KDE
    from scipy.stats import gaussian_kde
    if len(data) > 10:
        kde = gaussian_kde(data)
        x_range = np.linspace(data.min(), data.max(), 200)
        axes[i].plot(x_range, kde(x_range), color='black', linewidth=2, linestyle='--')
    
    axes[i].set_xlabel(labels[i], fontsize=12)
    axes[i].set_ylabel('Density', fontsize=12)
    axes[i].set_title(f'Distribution of {labels[i]}', fontsize=12, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    
    # Add statistics text
    stats_text = f'Mean: {data.mean():.3f}\nStd: {data.std():.3f}\nSkew: {data.skew():.3f}'
    axes[i].text(0.97, 0.97, stats_text, transform=axes[i].transAxes,
                 verticalalignment='top', horizontalalignment='right',
                 fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Box plots per cycle bin to show distribution evolution over battery life
df['cycle_bin'] = pd.cut(df['cycle_number'], bins=10, labels=False) + 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (col, label) in enumerate(zip(['voltage_V', 'current_A', 'temperature_C'],
                                      ['Voltage (V)', 'Current (A)', 'Temperature (\u00b0C)'])):
    if col in df.columns:
        sns.boxplot(data=df, x='cycle_bin', y=col, ax=axes[i], palette='viridis',
                    fliersize=1, linewidth=0.8)
        axes[i].set_xlabel('Cycle Bin (1=early, 10=late)', fontsize=11)
        axes[i].set_ylabel(label, fontsize=11)
        axes[i].set_title(f'{label} Distribution Over Battery Life', fontsize=12, fontweight='bold')
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Clean up temporary column
df.drop(columns=['cycle_bin'], inplace=True)

## 10. Correlation Analysis

We compute and visualize the Pearson correlation matrix among all numeric features. Strong correlations between measurement features and SOC indicate which features will be most informative for ML models.

In [ ]:
# Select numeric columns for correlation
corr_cols = [c for c in df.select_dtypes(include=[np.number]).columns 
             if c not in ['cell_id']]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9},
)

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Print correlations with SOC
if 'SOC_pct' in corr_matrix.columns:
    soc_corr = corr_matrix['SOC_pct'].drop('SOC_pct').sort_values(key=abs, ascending=False)
    print("\nFeature correlations with SOC (by absolute value):")
    print("=" * 45)
    for feat, val in soc_corr.items():
        bar = '+' * int(abs(val) * 20)
        print(f"  {feat:<20s}  {val:+.4f}  |{bar}")

## 11. Additional Explorations

In [ ]:
# Pair plot for key features (subsample for speed)
pair_cols = ['voltage_V', 'current_A', 'temperature_C']
if 'SOC_pct' in df.columns:
    pair_cols.append('SOC_pct')

available_pair = [c for c in pair_cols if c in df.columns]
sample_df = df[available_pair].dropna().sample(n=min(3000, len(df)), random_state=42)

g = sns.pairplot(sample_df, diag_kind='kde', plot_kws={'alpha': 0.3, 's': 8},
                 diag_kws={'linewidth': 1.5})
g.fig.suptitle('Pairwise Feature Relationships', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Voltage derivative analysis (dV/dt) - useful for SOC estimation
sample_cycle = df['cycle_number'].unique()[10]
cyc_df = df[df['cycle_number'] == sample_cycle].copy()
cyc_df['dV_dt'] = cyc_df['voltage_V'].diff() / cyc_df['time_s'].diff()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(cyc_df['time_s'], cyc_df['voltage_V'], color='#2196F3', linewidth=1.5)
axes[0].set_ylabel('Voltage (V)', fontsize=12)
axes[0].set_title(f'Voltage and dV/dt Analysis (Cycle {sample_cycle})', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cyc_df['time_s'], cyc_df['dV_dt'], color='#9C27B0', linewidth=1.0, alpha=0.7)
axes[1].set_ylabel('dV/dt (V/s)', fontsize=12)
axes[1].set_xlabel('Time (s)', fontsize=12)
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

# Clip extreme values for better visualization
dv_clip = cyc_df['dV_dt'].dropna().clip(
    cyc_df['dV_dt'].dropna().quantile(0.01),
    cyc_df['dV_dt'].dropna().quantile(0.99)
)
axes[1].set_ylim(dv_clip.min() * 1.2, dv_clip.max() * 1.2)

plt.tight_layout()
plt.show()

## 12. Key Observations Summary

### Dataset Characteristics
- The dataset contains discharge cycling data with voltage, current, and temperature measurements sampled at regular intervals.
- Each cycle consists of a constant-current discharge phase where voltage decreases monotonically from ~4.2V to ~2.7V.

### Voltage Behavior
- The discharge voltage curve shows the typical Li-ion profile: a relatively flat plateau region (~3.4-3.8V) followed by a steep drop near end of discharge.
- The voltage derivative (dV/dt) is a strong feature for SOC estimation, particularly in the plateau and knee regions.

### Capacity Degradation
- Clear capacity fade is observed over cycling, with approximately linear degradation in the initial cycles transitioning to slightly accelerated fade at later stages.
- SOH decreases progressively, consistent with SEI layer growth and loss of active lithium.

### Temperature Effects
- Temperature rises during discharge due to Joule heating (I\u00b2R losses) and electrochemical reactions.
- The temperature rise is correlated with current magnitude and SOC level.

### Feature Correlations
- Voltage has the strongest correlation with SOC, as expected from the OCV-SOC relationship.
- Temperature and time-within-cycle provide additional information for SOC estimation.
- Cycle number (aging indicator) affects the voltage-SOC mapping, motivating cycle-aware features.

### Implications for ML Modeling
- **Feature engineering:** Rolling statistics, voltage derivatives, and thermal features should improve model accuracy.
- **Train/test split:** Temporal splitting by cycle number is essential to prevent data leakage.
- **Normalization:** Feature scaling (StandardScaler) is recommended given the different physical units.
- **Sequence models:** The temporal nature of discharge data makes LSTM networks a natural choice alongside traditional regressors.